# Querying ERA5 with SQL

The following query computes the monthly area-weighted global mean of temperature over a year of ERA5 data.

It runs on AWS Lambda and then shows the cost of the run.

In [ ]:
import os

from zarr_datafusion_lambda import sql

# Where ERA5 and the workers are. Read once a kernel, at the first query.
os.environ["AWS_REGION"] = "us-east-1"

tables = {"era5": {"url": "s3://earthmover-icechunk-era5/icechunkV2", "group": "single/spatial"}}

In [ ]:
query = """
SELECT date_trunc('month', CAST(valid_time AS TIMESTAMP))              AS month,
       sum(t2m * cos(radians(latitude))) / sum(cos(radians(latitude))) AS mean_t2m
FROM era5
WHERE valid_time >= TIMESTAMP '2020-01-01' AND valid_time < TIMESTAMP '2021-01-01'
GROUP BY 1
ORDER BY 1
"""

In [ ]:
print(sql.explain(query, tables))

In [ ]:
result = sql.run(query, tables, progress=True, concurrency=990)
df = result.table.to_pandas()
print(f"{result.chunks} chunks in {result.invocations} invocations")
df

## Running in the client

A query fans out whenever the planner in zarr-datafusion-lambda can do so. 

Set the `lambda.enabled` to `false` to make the whole query run locally in the client.

The following compares running on lambda with running in the client on a point query over the time-series layout, which reads only a chunk or two.

In [ ]:
point_tables = {
    "era5": {"url": "s3://earthmover-icechunk-era5/icechunkV2", "group": "single/temporal"}
}
point_query = """
SELECT date_trunc('month', CAST(valid_time AS TIMESTAMP)) AS month,
       avg(t2m)                                          AS mean_t2m
FROM era5
WHERE latitude = 51.5 AND longitude = 0
  AND valid_time >= TIMESTAMP '2020-01-01' AND valid_time < TIMESTAMP '2021-01-01'
GROUP BY 1
ORDER BY 1
"""
in_client = {"lambda.enabled": "false"}

In [ ]:
print(sql.explain(point_query, point_tables))

In [ ]:
print(sql.explain(point_query, point_tables, config=in_client))

In [ ]:
%%time
result = sql.run(point_query, point_tables, progress=True)
df = result.table.to_pandas()
print(f"{result.chunks} chunks in {result.invocations} invocations")
df

In [ ]:
%%time
sql.run(point_query, point_tables, config=in_client).table.to_pandas()